# 04. Agregación de datos por barrios

## 1. Carga y comprobación de los datos

En este notebook se van a agrupar por barrios los anuncios que se limpiaron en el notebook anterior. Para ello, se utilizarán únicamente los anuncios que superaron todas las comprobaciones y que pueden considerarse válidos para analizar el mercado inmobiliario.

El objetivo es obtener una tabla con información representativa de cada uno de los 131 barrios oficiales de Madrid. Entre otros datos, se calcularán el precio habitual de las viviendas, el precio por metro cuadrado, la superficie, el número de habitaciones y la presencia de características como ascensor o vivienda exterior.

También se comprobará cuántos anuncios hay disponibles en cada barrio. Este punto es importante porque los resultados de un barrio con muchos anuncios serán, en principio, más representativos que los de otro en el que solo haya unos pocos.

Para representar el precio se utilizará principalmente la mediana, ya que se ve menos afectada por viviendas con precios especialmente altos o bajos. La media también se conservará como información complementaria.

Más adelante, esta tabla se combinará con los tiempos de desplazamiento y se utilizará como base para el sistema de recomendación.


In [71]:
# ============================================================
# Importaciones y configuración de rutas
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# El notebook puede ejecutarse:
#
# 1. desde la carpeta notebooks;
# 2. desde la raíz del proyecto.
#
# En ambos casos se localiza correctamente PROJECT_ROOT.

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


# ============================================================
# Archivos de entrada
# ============================================================

LISTINGS_CLEAN_PATH = (
    PROCESSED_DIR
    / "listings_clean.csv"
)

LISTINGS_MARKET_PATH = (
    PROCESSED_DIR
    / "listings_market_eligible.csv"
)

ZONES_PATH = (
    PROCESSED_DIR
    / "zones_master.csv"
)


input_paths = {
    "listings_clean": LISTINGS_CLEAN_PATH,
    "listings_market": LISTINGS_MARKET_PATH,
    "zones_master": ZONES_PATH,
}


# Comprobamos que todos los archivos necesarios existen antes
# de intentar cargarlos.

missing_files = [
    str(path)
    for path in input_paths.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "No se han encontrado los siguientes archivos:\n"
        + "\n".join(missing_files)
    )


# ============================================================
# Carga de los datasets
# ============================================================

listings_clean = pd.read_csv(
    LISTINGS_CLEAN_PATH,
    low_memory=False,
)

listings_market = pd.read_csv(
    LISTINGS_MARKET_PATH,
    low_memory=False,
)

zones = pd.read_csv(
    ZONES_PATH,
    low_memory=False,
)


# ============================================================
# Validación de las columnas necesarias
# ============================================================

required_clean_columns = {
    "Enlace",
    "zone_key_final",
    "use_in_neighborhood_model",
}

required_market_columns = {
    "Enlace",
    "zone_key_final",
    "price_eur",
    "price_per_m2",
    "area_m2",
    "rooms_clean",
    "bathrooms_clean",
    "elevator_binary",
    "exterior_binary",
}

required_zone_columns = {
    "zone_key",
    "neighborhood_name",
    "district_name",
}


def validate_required_columns(
    dataframe,
    required_columns,
    dataframe_name,
):
    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Faltan columnas en {dataframe_name}: "
            f"{sorted(missing_columns)}"
        )


validate_required_columns(
    listings_clean,
    required_clean_columns,
    "listings_clean",
)

validate_required_columns(
    listings_market,
    required_market_columns,
    "listings_market",
)

validate_required_columns(
    zones,
    required_zone_columns,
    "zones_master",
)


# ============================================================
# Conversión y comprobación de identificadores territoriales
# ============================================================

listings_clean["zone_key_final"] = (
    listings_clean["zone_key_final"]
    .astype("string")
)

listings_market["zone_key_final"] = (
    listings_market["zone_key_final"]
    .astype("string")
)

zones["zone_key"] = (
    zones["zone_key"]
    .astype("string")
)


# Los identificadores de los anuncios elegibles deben pertenecer
# a los 131 barrios oficiales.

unknown_market_zone_keys = sorted(
    set(
        listings_market[
            "zone_key_final"
        ].dropna()
    )
    - set(
        zones["zone_key"]
    )
)

if unknown_market_zone_keys:
    raise ValueError(
        "Existen zone_key_final que no aparecen en zones_master: "
        f"{unknown_market_zone_keys}"
    )


# ============================================================
# Comprobaciones de integridad
# ============================================================

EXPECTED_CLEAN_LISTINGS = 11826
EXPECTED_MARKET_LISTINGS = 11109
EXPECTED_OFFICIAL_ZONES = 131


assert len(listings_clean) == EXPECTED_CLEAN_LISTINGS, (
    "El número de anuncios del dataset limpio no coincide con "
    f"el esperado: {len(listings_clean)}."
)

assert len(listings_market) == EXPECTED_MARKET_LISTINGS, (
    "El número de anuncios elegibles no coincide con el "
    f"resultado del notebook 03: {len(listings_market)}."
)

assert len(zones) == EXPECTED_OFFICIAL_ZONES, (
    "La tabla territorial no contiene los 131 barrios oficiales."
)

assert zones["zone_key"].notna().all(), (
    "Existen barrios sin zone_key."
)

assert not zones["zone_key"].duplicated().any(), (
    "Existen zone_key duplicadas en zones_master."
)

assert listings_market["zone_key_final"].notna().all(), (
    "Existen anuncios elegibles sin barrio oficial."
)

assert listings_market["price_eur"].notna().all(), (
    "Existen anuncios elegibles sin precio."
)

assert listings_market["area_m2"].notna().all(), (
    "Existen anuncios elegibles sin superficie."
)

assert listings_market["price_per_m2"].notna().all(), (
    "Existen anuncios elegibles sin precio por metro cuadrado."
)


# Comprobamos que todos los anuncios del dataset elegible también
# aparecen en el dataset limpio.

missing_market_links = (
    set(listings_market["Enlace"])
    - set(listings_clean["Enlace"])
)

assert not missing_market_links, (
    "Hay anuncios elegibles que no aparecen en listings_clean."
)


# ============================================================
# Resumen de la carga
# ============================================================

print("Directorio del proyecto:")
print(PROJECT_ROOT)

print("\nArchivos cargados correctamente:")

print(
    f"- Anuncios limpios conservados: "
    f"{len(listings_clean):,}"
)

print(
    f"- Anuncios utilizables por barrio: "
    f"{len(listings_market):,}"
)

print(
    f"- Barrios oficiales: "
    f"{len(zones):,}"
)

print(
    "\nTodas las validaciones iniciales "
    "se han superado correctamente."
)

Directorio del proyecto:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas

Archivos cargados correctamente:
- Anuncios limpios conservados: 11,826
- Anuncios utilizables por barrio: 11,109
- Barrios oficiales: 131

Todas las validaciones iniciales se han superado correctamente.


## 2. Comprobación de la cobertura por barrios

Antes de calcular los valores representativos de cada barrio, conviene comprobar cómo se distribuyen los anuncios por Madrid.

En esta sección se contará cuántos anuncios originales había en cada barrio y cuántos continúan siendo utilizables después del proceso de limpieza. La diferencia entre ambos valores permitirá conocer cuántos anuncios se han excluido por ser duplicados, operaciones especiales, posibles errores o usos no residenciales.

También se identificarán los barrios que tienen pocos anuncios o que directamente no cuentan con información. Estos casos no deben eliminarse de la tabla, ya que el objetivo es conservar los 131 barrios oficiales, pero sí deben quedar claramente señalados.

Tener pocos anuncios no significa automáticamente que los datos sean incorrectos. Sin embargo, sí implica que las estimaciones del barrio pueden ser menos representativas, algo que se analizará más adelante mediante medidas de dispersión e incertidumbre.



In [72]:
# ============================================================
# Recuento de anuncios originales y elegibles por barrio
# ============================================================

# Número total de anuncios conservados en listings_clean.
# Incluye anuncios que después fueron excluidos del análisis por:
#
# - operaciones especiales;
# - usos no residenciales;
# - errores de superficie;
# - duplicidad;
# - otros criterios definidos en el notebook 03.

total_by_zone = (
    listings_clean
    .groupby(
        "zone_key_final",
        dropna=False,
    )
    .size()
    .rename("total_listings")
)


# Número de anuncios finalmente utilizables para calcular
# indicadores inmobiliarios por barrio.

eligible_by_zone = (
    listings_market
    .groupby(
        "zone_key_final",
        dropna=False,
    )
    .size()
    .rename("eligible_listings")
)


# ============================================================
# Incorporación de los 131 barrios oficiales
# ============================================================

coverage_by_zone = zones[
    [
        "zone_key",
        "neighborhood_name",
        "district_name",
    ]
].copy()


coverage_by_zone = coverage_by_zone.merge(
    total_by_zone,
    left_on="zone_key",
    right_index=True,
    how="left",
    validate="one_to_one",
)


coverage_by_zone = coverage_by_zone.merge(
    eligible_by_zone,
    left_on="zone_key",
    right_index=True,
    how="left",
    validate="one_to_one",
)


# Los barrios sin anuncios aparecen como NaN tras el merge.
# Se sustituyen por cero porque se trata de recuentos.

count_columns = [
    "total_listings",
    "eligible_listings",
]

coverage_by_zone[count_columns] = (
    coverage_by_zone[count_columns]
    .fillna(0)
    .astype(int)
)


# ============================================================
# Indicadores de cobertura y exclusión
# ============================================================

coverage_by_zone["excluded_listings"] = (
    coverage_by_zone["total_listings"]
    - coverage_by_zone["eligible_listings"]
)


# Porcentaje de anuncios originales que permanecen disponibles
# después del proceso de limpieza.

coverage_by_zone["eligible_percentage"] = np.where(
    coverage_by_zone["total_listings"].gt(0),
    (
        coverage_by_zone["eligible_listings"]
        / coverage_by_zone["total_listings"]
        * 100
    ),
    np.nan,
)


# Porcentaje de anuncios que fueron excluidos.

coverage_by_zone["excluded_percentage"] = np.where(
    coverage_by_zone["total_listings"].gt(0),
    (
        coverage_by_zone["excluded_listings"]
        / coverage_by_zone["total_listings"]
        * 100
    ),
    np.nan,
)


# Peso de cada barrio dentro del conjunto final de anuncios.

coverage_by_zone["market_share_percentage"] = (
    coverage_by_zone["eligible_listings"]
    / len(listings_market)
    * 100
)


# Estado de disponibilidad.
# Esto solo indica si existen datos, no su fiabilidad.

coverage_by_zone["coverage_status"] = np.select(
    [
        coverage_by_zone["total_listings"].eq(0),

        (
            coverage_by_zone["total_listings"].gt(0)
            & coverage_by_zone["eligible_listings"].eq(0)
        ),

        coverage_by_zone["eligible_listings"].gt(0),
    ],
    [
        "no_listings",
        "no_eligible_listings",
        "available",
    ],
    default="review",
)


# Redondeamos únicamente los porcentajes.
# Los recuentos permanecen como enteros.

percentage_columns = [
    "eligible_percentage",
    "excluded_percentage",
    "market_share_percentage",
]

coverage_by_zone[percentage_columns] = (
    coverage_by_zone[percentage_columns]
    .round(2)
)


# Orden territorial estable.

coverage_by_zone = (
    coverage_by_zone
    .sort_values(
        [
            "district_name",
            "neighborhood_name",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# Validaciones de integridad
# ============================================================

assert len(coverage_by_zone) == 131, (
    "La auditoría no contiene los 131 barrios oficiales."
)

assert not coverage_by_zone["zone_key"].duplicated().any(), (
    "Existen barrios duplicados en la auditoría."
)

assert (
    coverage_by_zone["total_listings"].sum()
    == len(listings_clean)
), (
    "El total de anuncios por barrio no coincide con "
    "listings_clean."
)

assert (
    coverage_by_zone["eligible_listings"].sum()
    == len(listings_market)
), (
    "El total de anuncios elegibles por barrio no coincide con "
    "listings_market."
)

assert (
    coverage_by_zone["eligible_listings"]
    <= coverage_by_zone["total_listings"]
).all(), (
    "Algún barrio contiene más anuncios elegibles que anuncios "
    "originales."
)

assert coverage_by_zone["excluded_listings"].ge(0).all(), (
    "Se han obtenido recuentos de exclusión negativos."
)

assert (
    coverage_by_zone["coverage_status"]
    != "review"
).all(), (
    "Existen barrios con un estado de cobertura no clasificado."
)


# ============================================================
# Resumen general
# ============================================================

num_zones_with_total_listings = (
    coverage_by_zone["total_listings"]
    .gt(0)
    .sum()
)

num_zones_with_eligible_listings = (
    coverage_by_zone["eligible_listings"]
    .gt(0)
    .sum()
)

num_zones_without_total_listings = (
    coverage_by_zone["total_listings"]
    .eq(0)
    .sum()
)

num_zones_without_eligible_listings = (
    coverage_by_zone["eligible_listings"]
    .eq(0)
    .sum()
)


print("Resumen de cobertura territorial:")

print(
    f"- Barrios oficiales: "
    f"{len(coverage_by_zone)}"
)

print(
    f"- Barrios con algún anuncio original: "
    f"{num_zones_with_total_listings}"
)

print(
    f"- Barrios con anuncios elegibles: "
    f"{num_zones_with_eligible_listings}"
)

print(
    f"- Barrios sin ningún anuncio original: "
    f"{num_zones_without_total_listings}"
)

print(
    f"- Barrios sin anuncios elegibles: "
    f"{num_zones_without_eligible_listings}"
)

print(
    f"- Anuncios originales agregados: "
    f"{coverage_by_zone['total_listings'].sum():,}"
)

print(
    f"- Anuncios elegibles agregados: "
    f"{coverage_by_zone['eligible_listings'].sum():,}"
)

print(
    f"- Anuncios excluidos: "
    f"{coverage_by_zone['excluded_listings'].sum():,}"
)

print(
    "\nDistribución del número de anuncios elegibles:"
)

display(
    coverage_by_zone[
        "eligible_listings"
    ].describe()
)

Resumen de cobertura territorial:
- Barrios oficiales: 131
- Barrios con algún anuncio original: 129
- Barrios con anuncios elegibles: 129
- Barrios sin ningún anuncio original: 2
- Barrios sin anuncios elegibles: 2
- Anuncios originales agregados: 11,826
- Anuncios elegibles agregados: 11,109
- Anuncios excluidos: 717

Distribución del número de anuncios elegibles:


count    131.000000
mean      84.801527
std       96.483910
min        0.000000
25%       27.500000
50%       60.000000
75%       98.000000
max      601.000000
Name: eligible_listings, dtype: float64


### 2.1. Barrios con pocos datos o sin anuncios

A continuación, se revisarán los barrios que no tienen ningún anuncio válido y aquellos que cuentan con menos información.

También se observarán los barrios en los que se ha excluido una proporción elevada de anuncios. Esto permitirá comprobar si las reglas de limpieza han afectado de forma especial a alguna zona concreta.

Esta revisión servirá para detectar posibles limitaciones de los datos antes de calcular los indicadores inmobiliarios definitivos.


In [73]:
# ============================================================
# Barrios sin anuncios elegibles
# ============================================================

zones_without_eligible_listings = (
    coverage_by_zone.loc[
        coverage_by_zone[
            "eligible_listings"
        ].eq(0),
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
            "total_listings",
            "eligible_listings",
            "excluded_listings",
            "coverage_status",
        ],
    ]
    .sort_values(
        [
            "district_name",
            "neighborhood_name",
        ]
    )
    .reset_index(drop=True)
)


print("Barrios sin anuncios elegibles:")

if zones_without_eligible_listings.empty:
    print(
        "Todos los barrios disponen de al menos "
        "un anuncio elegible."
    )
else:
    display(zones_without_eligible_listings)


# ============================================================
# Barrios con menor número de anuncios elegibles
# ============================================================

zones_with_eligible_listings = (
    coverage_by_zone.loc[
        coverage_by_zone[
            "eligible_listings"
        ].gt(0)
    ]
    .copy()
)


lowest_coverage_zones = (
    zones_with_eligible_listings[
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
            "total_listings",
            "eligible_listings",
            "excluded_listings",
            "eligible_percentage",
            "market_share_percentage",
        ]
    ]
    .sort_values(
        [
            "eligible_listings",
            "eligible_percentage",
            "district_name",
            "neighborhood_name",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
    )
    .head(20)
    .reset_index(drop=True)
)


print(
    "\nVeinte barrios con menor número "
    "de anuncios elegibles:"
)

display(lowest_coverage_zones)


# ============================================================
# Barrios con mayor porcentaje de exclusiones
# ============================================================

# Exigimos un mínimo de diez anuncios originales para evitar
# que un porcentaje elevado dependa únicamente de uno o dos
# registros.

MIN_TOTAL_LISTINGS_FOR_EXCLUSION_REVIEW = 10


highest_exclusion_zones = (
    coverage_by_zone.loc[
        coverage_by_zone[
            "total_listings"
        ].ge(
            MIN_TOTAL_LISTINGS_FOR_EXCLUSION_REVIEW
        ),
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
            "total_listings",
            "eligible_listings",
            "excluded_listings",
            "excluded_percentage",
        ],
    ]
    .sort_values(
        [
            "excluded_percentage",
            "excluded_listings",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(20)
    .reset_index(drop=True)
)


print(
    "\nVeinte barrios con mayor porcentaje de exclusiones "
    f"(mínimo {MIN_TOTAL_LISTINGS_FOR_EXCLUSION_REVIEW} "
    "anuncios originales):"
)

display(highest_exclusion_zones)

Barrios sin anuncios elegibles:


,zone_key,neighborhood_name,district_name,total_listings,eligible_listings,excluded_listings,coverage_status
0,MAD-027,Atocha,Arganzuela,0,0,0,no_listings
1,MAD-192,Valdebernardo,Vicálvaro,0,0,0,no_listings



Veinte barrios con menor número de anuncios elegibles:


,zone_key,neighborhood_name,district_name,total_listings,eligible_listings,excluded_listings,eligible_percentage,market_share_percentage
0,MAD-142,Horcajo,Moratalaz,2,2,0,100.00,0.02
1,MAD-212,Aeropuerto,Barajas,4,3,1,75.00,0.03
2,MAD-141,Pavones,Moratalaz,5,4,1,80.00,0.04
3,MAD-081,El Pardo,Fuencarral - El Pardo,5,5,0,100.00,0.05
4,MAD-158,Atalaya,Ciudad Lineal,8,8,0,100.00,0.07
5,MAD-193,Valderrivas,Vicálvaro,10,10,0,100.00,0.09
6,MAD-088,El Goloso,Fuencarral - El Pardo,12,12,0,100.00,0.11
7,MAD-165,Apóstol Santiago,Hortaleza,12,12,0,100.00,0.11
8,MAD-106,Cuatro Vientos,Latina,15,15,0,100.00,0.14
9,MAD-203,Amposta,San Blas - Canillejas,25,16,9,64.00,0.14



Veinte barrios con mayor porcentaje de exclusiones (mínimo 10 anuncios originales):


,zone_key,neighborhood_name,district_name,total_listings,eligible_listings,excluded_listings,excluded_percentage
0,MAD-172,San Cristóbal,Villaverde,40,22,18,45.00
1,MAD-203,Amposta,San Blas - Canillejas,25,16,9,36.00
2,MAD-174,Los Rosales,Villaverde,58,40,18,31.03
3,MAD-101,Los Cármenes,Latina,34,24,10,29.41
4,MAD-204,Arcos,San Blas - Canillejas,29,21,8,27.59
5,MAD-117,Abrantes,Carabanchel,55,40,15,27.27
6,MAD-121,Orcasitas,Usera,26,19,7,26.92
7,MAD-115,Puerta Bonita,Carabanchel,101,75,26,25.74
8,MAD-175,Ángeles,Villaverde,40,30,10,25.00
9,MAD-173,Butarque,Villaverde,24,18,6,25.00


## 3. Cálculo de los datos representativos de cada barrio

Una vez comprobado cuántos anuncios hay disponibles en cada zona, se pueden calcular los valores que describen el mercado inmobiliario de cada barrio.

Para los precios, se calcularán tanto la media como la mediana. La mediana será el valor principal, ya que representa mejor el precio habitual de una zona cuando existen viviendas especialmente caras o baratas que pueden alterar la media.

También se calcularán datos sobre la superficie, el número de habitaciones y baños, y la proporción de viviendas con ascensor o exteriores.

No todos los anuncios incluyen todas estas características. Por este motivo, además del valor calculado, se guardará el número de anuncios que realmente contienen información sobre cada atributo.


In [74]:
# ============================================================
# Agrupación de los anuncios por barrio
# ============================================================

zone_group = listings_market.groupby(
    "zone_key_final",
    sort=True,
)


# ============================================================
# Cálculo de los indicadores básicos
# ============================================================

neighborhood_statistics = zone_group.agg(
    # Número de anuncios disponibles
    num_listings=(
        "Enlace",
        "nunique",
    ),

    # Precio total de las viviendas
    price_eur_mean=(
        "price_eur",
        "mean",
    ),
    price_eur_median=(
        "price_eur",
        "median",
    ),
    price_eur_min=(
        "price_eur",
        "min",
    ),
    price_eur_max=(
        "price_eur",
        "max",
    ),

    # Precio por metro cuadrado
    price_m2_mean=(
        "price_per_m2",
        "mean",
    ),
    price_m2_median=(
        "price_per_m2",
        "median",
    ),
    price_m2_std=(
        "price_per_m2",
        "std",
    ),

    # Superficie
    area_m2_mean=(
        "area_m2",
        "mean",
    ),
    area_m2_median=(
        "area_m2",
        "median",
    ),
    area_m2_min=(
        "area_m2",
        "min",
    ),
    area_m2_max=(
        "area_m2",
        "max",
    ),

    # Habitaciones
    rooms_mean=(
        "rooms_clean",
        "mean",
    ),
    rooms_median=(
        "rooms_clean",
        "median",
    ),
    rooms_known_count=(
        "rooms_clean",
        "count",
    ),

    # Baños
    bathrooms_mean=(
        "bathrooms_clean",
        "mean",
    ),
    bathrooms_median=(
        "bathrooms_clean",
        "median",
    ),
    bathrooms_known_count=(
        "bathrooms_clean",
        "count",
    ),

    # Ascensor
    elevator_ratio=(
        "elevator_binary",
        "mean",
    ),
    elevator_known_count=(
        "elevator_binary",
        "count",
    ),

    # Vivienda exterior
    exterior_ratio=(
        "exterior_binary",
        "mean",
    ),
    exterior_known_count=(
        "exterior_binary",
        "count",
    ),
).reset_index()


# Cambiamos el nombre para utilizar el mismo identificador
# territorial que aparece en zones_master.

neighborhood_statistics = (
    neighborhood_statistics.rename(
        columns={
            "zone_key_final": "zone_key",
        }
    )
)


# ============================================================
# Redondeo de los resultados descriptivos
# ============================================================

currency_columns = [
    "price_eur_mean",
    "price_eur_median",
    "price_eur_min",
    "price_eur_max",
]

price_m2_columns = [
    "price_m2_mean",
    "price_m2_median",
    "price_m2_std",
]

area_columns = [
    "area_m2_mean",
    "area_m2_median",
    "area_m2_min",
    "area_m2_max",
]

room_columns = [
    "rooms_mean",
    "rooms_median",
    "bathrooms_mean",
    "bathrooms_median",
]

ratio_columns = [
    "elevator_ratio",
    "exterior_ratio",
]


neighborhood_statistics[currency_columns] = (
    neighborhood_statistics[currency_columns]
    .round(2)
)

neighborhood_statistics[price_m2_columns] = (
    neighborhood_statistics[price_m2_columns]
    .round(2)
)

neighborhood_statistics[area_columns] = (
    neighborhood_statistics[area_columns]
    .round(2)
)

neighborhood_statistics[room_columns] = (
    neighborhood_statistics[room_columns]
    .round(2)
)

neighborhood_statistics[ratio_columns] = (
    neighborhood_statistics[ratio_columns]
    .round(4)
)


# ============================================================
# Comprobaciones
# ============================================================

assert not neighborhood_statistics[
    "zone_key"
].duplicated().any(), (
    "Se ha obtenido más de una fila para algún barrio."
)

assert (
    neighborhood_statistics["num_listings"].sum()
    == len(listings_market)
), (
    "El total de anuncios agregados no coincide con "
    "listings_market."
)

assert (
    neighborhood_statistics["rooms_known_count"]
    <= neighborhood_statistics["num_listings"]
).all(), (
    "El recuento de habitaciones conocidas no es válido."
)

assert (
    neighborhood_statistics["bathrooms_known_count"]
    <= neighborhood_statistics["num_listings"]
).all(), (
    "El recuento de baños conocidos no es válido."
)

assert (
    neighborhood_statistics["elevator_known_count"]
    <= neighborhood_statistics["num_listings"]
).all(), (
    "El recuento de información sobre ascensor no es válido."
)

assert (
    neighborhood_statistics["exterior_known_count"]
    <= neighborhood_statistics["num_listings"]
).all(), (
    "El recuento de información sobre exterior no es válido."
)

assert (
    neighborhood_statistics["elevator_ratio"]
    .dropna()
    .between(0, 1)
    .all()
), (
    "Se han obtenido proporciones de ascensor fuera del "
    "intervalo entre 0 y 1."
)

assert (
    neighborhood_statistics["exterior_ratio"]
    .dropna()
    .between(0, 1)
    .all()
), (
    "Se han obtenido proporciones de viviendas exteriores "
    "fuera del intervalo entre 0 y 1."
)


# ============================================================
# Resumen
# ============================================================

print(
    "Barrios con estadísticas inmobiliarias:",
    len(neighborhood_statistics),
)

print(
    "Anuncios incluidos en la agregación:",
    neighborhood_statistics[
        "num_listings"
    ].sum(),
)

print(
    "\nPrimeras filas de las estadísticas:"
)

display(
    neighborhood_statistics.head()
)

Barrios con estadísticas inmobiliarias: 129
Anuncios incluidos en la agregación: 11109

Primeras filas de las estadísticas:


,zone_key,num_listings,price_eur_mean,price_eur_median,price_eur_min,price_eur_max,price_m2_mean,price_m2_median,price_m2_std,area_m2_mean,...,rooms_mean,rooms_median,rooms_known_count,bathrooms_mean,bathrooms_median,bathrooms_known_count,elevator_ratio,elevator_known_count,exterior_ratio,exterior_known_count
0,MAD-011,297,771118.19,540000.0,150000,15000000,7003.76,6614.32,2142.05,104.48,...,2.32,2.0,280,1.68,2.0,59,0.7162,296,0.7306,297
1,MAD-012,349,415116.23,310000.0,79900,2950000,5748.80,5566.67,1553.56,75.71,...,2.21,2.0,299,1.43,1.0,67,0.5318,346,0.5777,341
2,MAD-013,173,1137456.01,800000.0,185000,5200000,8157.50,7755.56,2517.31,136.10,...,2.61,2.0,171,2.16,2.0,32,0.7457,173,0.8294,170
3,MAD-014,273,1362072.53,1079000.0,169000,4890000,9295.85,9076.92,2497.12,143.25,...,2.75,3.0,269,2.03,2.0,68,0.8498,273,0.8529,272
4,MAD-015,439,797022.99,690000.0,136000,3700000,7782.57,7571.43,2291.90,102.33,...,2.56,2.0,405,1.76,2.0,96,0.6819,437,0.7609,435


### 3.1. Diferencias de precios dentro de cada barrio

La mediana nos permite conocer el precio más representativo de cada barrio, pero no nos dice si todas las viviendas tienen precios parecidos o si existe mucha diferencia entre unas y otras.

Para verlo, se calcularán el primer y el tercer cuartil. Entre estos dos valores se encuentra la mitad central de los anuncios del barrio, dejando fuera los precios más bajos y los más altos.

También se calculará el rango intercuartílico, que mide la distancia entre ambos cuartiles. Cuanto mayor sea este valor, mayor será la variedad de precios dentro del barrio.

Estos cálculos se realizarán para el precio total, el precio por metro cuadrado y la superficie de las viviendas.

In [75]:
# ============================================================
# Cuartiles por barrio
# ============================================================

# Q1 deja por debajo al 25 % de los anuncios.
# Q3 deja por debajo al 75 % de los anuncios.
# Entre ambos valores se encuentra la mitad central de los datos.

neighborhood_quartiles = zone_group.agg(
    # Precio total
    price_eur_q1=(
        "price_eur",
        lambda values: values.quantile(0.25),
    ),
    price_eur_q3=(
        "price_eur",
        lambda values: values.quantile(0.75),
    ),

    # Precio por metro cuadrado
    price_m2_q1=(
        "price_per_m2",
        lambda values: values.quantile(0.25),
    ),
    price_m2_q3=(
        "price_per_m2",
        lambda values: values.quantile(0.75),
    ),

    # Superficie
    area_m2_q1=(
        "area_m2",
        lambda values: values.quantile(0.25),
    ),
    area_m2_q3=(
        "area_m2",
        lambda values: values.quantile(0.75),
    ),
).reset_index()


neighborhood_quartiles = (
    neighborhood_quartiles.rename(
        columns={
            "zone_key_final": "zone_key",
        }
    )
)


# ============================================================
# Incorporación de los cuartiles a la tabla principal
# ============================================================

neighborhood_statistics = (
    neighborhood_statistics.merge(
        neighborhood_quartiles,
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# Rangos intercuartílicos
# ============================================================

# El rango intercuartílico es la diferencia entre Q3 y Q1.
# Representa la amplitud de la mitad central de los anuncios.

neighborhood_statistics["price_eur_iqr"] = (
    neighborhood_statistics["price_eur_q3"]
    - neighborhood_statistics["price_eur_q1"]
)

neighborhood_statistics["price_m2_iqr"] = (
    neighborhood_statistics["price_m2_q3"]
    - neighborhood_statistics["price_m2_q1"]
)

neighborhood_statistics["area_m2_iqr"] = (
    neighborhood_statistics["area_m2_q3"]
    - neighborhood_statistics["area_m2_q1"]
)


# La dispersión relativa permite comparar barrios baratos y caros.
# Indica qué porcentaje representa el IQR respecto a la mediana.

neighborhood_statistics[
    "price_m2_relative_iqr_percentage"
] = np.where(
    neighborhood_statistics[
        "price_m2_median"
    ].gt(0),
    (
        neighborhood_statistics[
            "price_m2_iqr"
        ]
        / neighborhood_statistics[
            "price_m2_median"
        ]
        * 100
    ),
    np.nan,
)


# ============================================================
# Redondeo
# ============================================================

quartile_currency_columns = [
    "price_eur_q1",
    "price_eur_q3",
    "price_eur_iqr",
]

quartile_price_m2_columns = [
    "price_m2_q1",
    "price_m2_q3",
    "price_m2_iqr",
]

quartile_area_columns = [
    "area_m2_q1",
    "area_m2_q3",
    "area_m2_iqr",
]


neighborhood_statistics[
    quartile_currency_columns
] = (
    neighborhood_statistics[
        quartile_currency_columns
    ].round(2)
)

neighborhood_statistics[
    quartile_price_m2_columns
] = (
    neighborhood_statistics[
        quartile_price_m2_columns
    ].round(2)
)

neighborhood_statistics[
    quartile_area_columns
] = (
    neighborhood_statistics[
        quartile_area_columns
    ].round(2)
)

neighborhood_statistics[
    "price_m2_relative_iqr_percentage"
] = (
    neighborhood_statistics[
        "price_m2_relative_iqr_percentage"
    ].round(2)
)


# ============================================================
# Validaciones
# ============================================================

assert (
    neighborhood_statistics["price_eur_q1"]
    <= neighborhood_statistics["price_eur_median"]
).all(), (
    "El primer cuartil del precio supera a la mediana."
)

assert (
    neighborhood_statistics["price_eur_median"]
    <= neighborhood_statistics["price_eur_q3"]
).all(), (
    "La mediana del precio supera al tercer cuartil."
)

assert (
    neighborhood_statistics["price_m2_q1"]
    <= neighborhood_statistics["price_m2_median"]
).all(), (
    "El primer cuartil del precio por m² supera a la mediana."
)

assert (
    neighborhood_statistics["price_m2_median"]
    <= neighborhood_statistics["price_m2_q3"]
).all(), (
    "La mediana del precio por m² supera al tercer cuartil."
)

assert (
    neighborhood_statistics["area_m2_q1"]
    <= neighborhood_statistics["area_m2_median"]
).all(), (
    "El primer cuartil de superficie supera a la mediana."
)

assert (
    neighborhood_statistics["area_m2_median"]
    <= neighborhood_statistics["area_m2_q3"]
).all(), (
    "La mediana de superficie supera al tercer cuartil."
)

assert (
    neighborhood_statistics["price_eur_iqr"]
    .ge(0)
    .all()
), (
    "Se han obtenido rangos intercuartílicos negativos."
)

assert (
    neighborhood_statistics["price_m2_iqr"]
    .ge(0)
    .all()
), (
    "Se han obtenido rangos intercuartílicos negativos."
)

assert (
    neighborhood_statistics["area_m2_iqr"]
    .ge(0)
    .all()
), (
    "Se han obtenido rangos intercuartílicos negativos."
)


# ============================================================
# Revisión de los resultados
# ============================================================

print(
    "Cuartiles calculados para:",
    len(neighborhood_quartiles),
    "barrios",
)

print(
    "\nBarrios con mayor dispersión relativa "
    "en el precio por metro cuadrado:"
)

display(
    neighborhood_statistics[
        [
            "zone_key",
            "num_listings",
            "price_m2_q1",
            "price_m2_median",
            "price_m2_q3",
            "price_m2_iqr",
            "price_m2_relative_iqr_percentage",
        ]
    ]
    .sort_values(
        "price_m2_relative_iqr_percentage",
        ascending=False,
    )
    .head(15)
)

Cuartiles calculados para: 129 barrios

Barrios con mayor dispersión relativa en el precio por metro cuadrado:


,zone_key,num_listings,price_m2_q1,price_m2_median,price_m2_q3,price_m2_iqr,price_m2_relative_iqr_percentage
55,MAD-096,74,2552.39,3580.19,4994.05,2441.66,68.20
44,MAD-083,136,3823.01,5459.21,7521.51,3698.50,67.75
93,MAD-154,75,3459.91,4487.18,6084.49,2624.59,58.49
115,MAD-194,75,2261.84,3670.21,4356.67,2094.82,57.08
97,MAD-158,8,4656.96,7637.49,8902.78,4245.82,55.59
103,MAD-165,12,3500.72,4203.28,5822.16,2321.44,55.23
95,MAD-156,33,4845.36,6508.47,8049.38,3204.02,49.23
72,MAD-122,38,2584.56,3411.29,4234.23,1649.67,48.36
2,MAD-013,173,6321.84,7755.56,9900.00,3578.16,46.14
127,MAD-214,17,3639.13,4241.25,5591.40,1952.27,46.03


### 3.2. Comprobar cuánta información hay sobre cada característica

El precio y la superficie están disponibles en todos los anuncios utilizados, pero no ocurre lo mismo con el número de habitaciones, los baños, el ascensor o si la vivienda es exterior.

Por este motivo, no basta con calcular el valor medio de cada característica. También es necesario indicar cuántos anuncios contienen realmente esa información.

Por ejemplo, una media de baños calculada a partir de cinco anuncios no tiene la misma representatividad que otra calculada con cien. Los porcentajes de cobertura permitirán saber cuándo una característica puede utilizarse con confianza y cuándo debe interpretarse con precaución.

In [76]:
# ============================================================
# Cobertura de las características de las viviendas
# ============================================================

attribute_count_columns = {
    "rooms": "rooms_known_count",
    "bathrooms": "bathrooms_known_count",
    "elevator": "elevator_known_count",
    "exterior": "exterior_known_count",
}


for attribute, known_column in (
    attribute_count_columns.items()
):
    unknown_column = (
        f"{attribute}_unknown_count"
    )

    coverage_column = (
        f"{attribute}_coverage_percentage"
    )

    # Número de anuncios que no incluyen el dato.
    neighborhood_statistics[
        unknown_column
    ] = (
        neighborhood_statistics["num_listings"]
        - neighborhood_statistics[known_column]
    )

    # Porcentaje de anuncios del barrio que sí contienen el dato.
    neighborhood_statistics[
        coverage_column
    ] = (
        neighborhood_statistics[known_column]
        / neighborhood_statistics["num_listings"]
        * 100
    ).round(2)


# ============================================================
# Validaciones
# ============================================================

for attribute, known_column in (
    attribute_count_columns.items()
):
    unknown_column = (
        f"{attribute}_unknown_count"
    )

    coverage_column = (
        f"{attribute}_coverage_percentage"
    )

    assert (
        neighborhood_statistics[known_column]
        + neighborhood_statistics[unknown_column]
        == neighborhood_statistics["num_listings"]
    ).all(), (
        f"Los recuentos conocidos y desconocidos de "
        f"{attribute} no coinciden con el total."
    )

    assert (
        neighborhood_statistics[unknown_column]
        .ge(0)
        .all()
    ), (
        f"Se han obtenido valores desconocidos negativos "
        f"para {attribute}."
    )

    assert (
        neighborhood_statistics[coverage_column]
        .between(0, 100)
        .all()
    ), (
        f"La cobertura de {attribute} contiene valores "
        "fuera del intervalo entre 0 y 100."
    )


# ============================================================
# Resumen de la cobertura
# ============================================================

attribute_coverage_summary = pd.DataFrame(
    {
        "attribute": [
            "Habitaciones",
            "Baños",
            "Ascensor",
            "Vivienda exterior",
        ],

        "median_coverage_percentage": [
            neighborhood_statistics[
                "rooms_coverage_percentage"
            ].median(),

            neighborhood_statistics[
                "bathrooms_coverage_percentage"
            ].median(),

            neighborhood_statistics[
                "elevator_coverage_percentage"
            ].median(),

            neighborhood_statistics[
                "exterior_coverage_percentage"
            ].median(),
        ],

        "minimum_coverage_percentage": [
            neighborhood_statistics[
                "rooms_coverage_percentage"
            ].min(),

            neighborhood_statistics[
                "bathrooms_coverage_percentage"
            ].min(),

            neighborhood_statistics[
                "elevator_coverage_percentage"
            ].min(),

            neighborhood_statistics[
                "exterior_coverage_percentage"
            ].min(),
        ],

        "maximum_coverage_percentage": [
            neighborhood_statistics[
                "rooms_coverage_percentage"
            ].max(),

            neighborhood_statistics[
                "bathrooms_coverage_percentage"
            ].max(),

            neighborhood_statistics[
                "elevator_coverage_percentage"
            ].max(),

            neighborhood_statistics[
                "exterior_coverage_percentage"
            ].max(),
        ],
    }
)


coverage_summary_columns = [
    "median_coverage_percentage",
    "minimum_coverage_percentage",
    "maximum_coverage_percentage",
]

attribute_coverage_summary[
    coverage_summary_columns
] = (
    attribute_coverage_summary[
        coverage_summary_columns
    ].round(2)
)


print(
    "Resumen de la cobertura de las características:"
)

display(attribute_coverage_summary)


print(
    "\nBarrios con menor cobertura de baños:"
)

display(
    neighborhood_statistics[
        [
            "zone_key",
            "num_listings",
            "bathrooms_known_count",
            "bathrooms_unknown_count",
            "bathrooms_coverage_percentage",
            "bathrooms_mean",
            "bathrooms_median",
        ]
    ]
    .sort_values(
        [
            "bathrooms_coverage_percentage",
            "num_listings",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .head(15)
)

Resumen de la cobertura de las características:


,attribute,median_coverage_percentage,minimum_coverage_percentage,maximum_coverage_percentage
0,Habitaciones,98.53,66.67,100.0
1,Baños,25.00,0.00,62.5
2,Ascensor,97.14,1.35,100.0
3,Vivienda exterior,93.88,9.46,100.0



Barrios con menor cobertura de baños:


,zone_key,num_listings,bathrooms_known_count,bathrooms_unknown_count,bathrooms_coverage_percentage,bathrooms_mean,bathrooms_median
62,MAD-106,15,0,15,0.00,NaN,NaN
125,MAD-212,3,0,3,0.00,NaN,NaN
115,MAD-194,75,6,69,8.00,2.17,2.0
111,MAD-182,24,2,22,8.33,1.00,1.0
57,MAD-101,24,3,21,12.50,2.33,3.0
116,MAD-201,91,12,79,13.19,1.50,1.5
123,MAD-208,29,4,25,13.79,2.00,1.5
100,MAD-162,103,15,88,14.56,2.53,2.0
76,MAD-126,27,4,23,14.81,1.25,1.0
55,MAD-096,74,11,63,14.86,4.09,3.0


### 3.3. Comprobar la estabilidad del precio de cada barrio

La mediana del precio por metro cuadrado es el valor principal que se utilizará para representar el coste de cada barrio. Sin embargo, esta mediana no tiene la misma estabilidad en todos los casos.

En un barrio con muchos anuncios y precios parecidos, el valor calculado debería cambiar poco si se utilizara otra muestra de viviendas. En cambio, cuando hay pocos anuncios o los precios son muy diferentes entre sí, la mediana puede variar bastante.

Para medir esta incertidumbre se utilizará bootstrap. Este método crea muchas muestras nuevas a partir de los anuncios disponibles en cada barrio y vuelve a calcular la mediana en cada una de ellas.

A partir de estas repeticiones se obtendrá un intervalo de confianza del 95 %. Un intervalo estrecho indicará que la mediana es bastante estable, mientras que un intervalo amplio indicará que debe interpretarse con mayor precaución.

En los barrios con menos de cinco anuncios no se calculará este intervalo, ya que una muestra tan pequeña podría producir una sensación de precisión engañosa.

In [77]:
# ============================================================
# Intervalos de confianza bootstrap de la mediana del precio/m²
# ============================================================

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
MIN_BOOTSTRAP_SAMPLE_SIZE = 5
BOOTSTRAP_RANDOM_SEED = 42


def calculate_bootstrap_median_interval(
    values,
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    random_seed=BOOTSTRAP_RANDOM_SEED,
):
    """
    Calcula mediante bootstrap el intervalo de confianza de la
    mediana de una serie de valores.

    Si existen menos de cinco observaciones, no se calcula el
    intervalo para evitar resultados poco representativos.
    """

    clean_values = (
        pd.to_numeric(
            pd.Series(values),
            errors="coerce",
        )
        .dropna()
        .to_numpy(dtype=float)
    )

    sample_size = len(clean_values)

    if sample_size < MIN_BOOTSTRAP_SAMPLE_SIZE:
        return {
            "bootstrap_sample_size": sample_size,
            "price_m2_median_ci_low": np.nan,
            "price_m2_median_ci_high": np.nan,
            "price_m2_median_ci_width": np.nan,
            "price_m2_median_bootstrap_se": np.nan,
            "bootstrap_status": "insufficient_sample",
        }


    rng = np.random.default_rng(
        random_seed
    )


    # Cada fila representa una nueva muestra creada con
    # reemplazo a partir de los anuncios originales del barrio.

    bootstrap_samples = rng.choice(
        clean_values,
        size=(
            n_bootstrap,
            sample_size,
        ),
        replace=True,
    )


    bootstrap_medians = np.median(
        bootstrap_samples,
        axis=1,
    )


    alpha = 1 - confidence_level

    lower_limit, upper_limit = np.quantile(
        bootstrap_medians,
        [
            alpha / 2,
            1 - alpha / 2,
        ],
    )


    return {
        "bootstrap_sample_size": sample_size,
        "price_m2_median_ci_low": lower_limit,
        "price_m2_median_ci_high": upper_limit,
        "price_m2_median_ci_width": (
            upper_limit - lower_limit
        ),
        "price_m2_median_bootstrap_se": (
            bootstrap_medians.std(ddof=1)
        ),
        "bootstrap_status": "calculated",
    }

In [78]:
# ============================================================
# Aplicación del bootstrap a cada barrio
# ============================================================

bootstrap_results = []


for position, (
    zone_key,
    zone_data,
) in enumerate(
    zone_group["price_per_m2"]
):
    # Se utiliza una semilla diferente pero reproducible
    # para cada barrio.

    zone_seed = (
        BOOTSTRAP_RANDOM_SEED
        + position
    )


    result = (
        calculate_bootstrap_median_interval(
            values=zone_data,
            n_bootstrap=BOOTSTRAP_ITERATIONS,
            confidence_level=(
                BOOTSTRAP_CONFIDENCE_LEVEL
            ),
            random_seed=zone_seed,
        )
    )


    result["zone_key"] = zone_key
    result["bootstrap_iterations"] = (
        BOOTSTRAP_ITERATIONS
    )
    result["bootstrap_confidence_level"] = (
        BOOTSTRAP_CONFIDENCE_LEVEL
    )


    bootstrap_results.append(result)


bootstrap_intervals = pd.DataFrame(
    bootstrap_results
)


# Colocamos zone_key como primera columna.

bootstrap_columns_order = [
    "zone_key",
    "bootstrap_sample_size",
    "bootstrap_iterations",
    "bootstrap_confidence_level",
    "price_m2_median_ci_low",
    "price_m2_median_ci_high",
    "price_m2_median_ci_width",
    "price_m2_median_bootstrap_se",
    "bootstrap_status",
]

bootstrap_intervals = bootstrap_intervals[
    bootstrap_columns_order
]


# ============================================================
# Incorporación a las estadísticas del barrio
# ============================================================

neighborhood_statistics = (
    neighborhood_statistics.merge(
        bootstrap_intervals,
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
)


# Anchura del intervalo respecto a la mediana.
# Este porcentaje permitirá comparar barrios con precios
# diferentes.

neighborhood_statistics[
    "price_m2_median_ci_relative_width_percentage"
] = np.where(
    (
        neighborhood_statistics[
            "bootstrap_status"
        ].eq("calculated")
        & neighborhood_statistics[
            "price_m2_median"
        ].gt(0)
    ),
    (
        neighborhood_statistics[
            "price_m2_median_ci_width"
        ]
        / neighborhood_statistics[
            "price_m2_median"
        ]
        * 100
    ),
    np.nan,
)


# ============================================================
# Redondeo
# ============================================================

bootstrap_numeric_columns = [
    "price_m2_median_ci_low",
    "price_m2_median_ci_high",
    "price_m2_median_ci_width",
    "price_m2_median_bootstrap_se",
    "price_m2_median_ci_relative_width_percentage",
]


neighborhood_statistics[
    bootstrap_numeric_columns
] = (
    neighborhood_statistics[
        bootstrap_numeric_columns
    ].round(2)
)


# ============================================================
# Validaciones
# ============================================================

assert not bootstrap_intervals[
    "zone_key"
].duplicated().any(), (
    "Se ha calculado más de un intervalo para algún barrio."
)

assert len(bootstrap_intervals) == len(
    neighborhood_statistics
), (
    "No se han calculado resultados bootstrap para todos "
    "los barrios con anuncios."
)

assert (
    neighborhood_statistics[
        "bootstrap_sample_size"
    ]
    == neighborhood_statistics[
        "num_listings"
    ]
).all(), (
    "El tamaño utilizado por bootstrap no coincide con el "
    "número de anuncios del barrio."
)


calculated_bootstrap_mask = (
    neighborhood_statistics[
        "bootstrap_status"
    ].eq("calculated")
)


assert (
    neighborhood_statistics.loc[
        calculated_bootstrap_mask,
        "price_m2_median_ci_low",
    ]
    <= neighborhood_statistics.loc[
        calculated_bootstrap_mask,
        "price_m2_median_ci_high",
    ]
).all(), (
    "Se han obtenido intervalos de confianza incorrectos."
)


assert (
    neighborhood_statistics.loc[
        calculated_bootstrap_mask,
        "price_m2_median_ci_width",
    ]
    .ge(0)
    .all()
), (
    "Se han obtenido intervalos con anchura negativa."
)


assert (
    neighborhood_statistics.loc[
        calculated_bootstrap_mask,
        "price_m2_median_ci_relative_width_percentage",
    ]
    .ge(0)
    .all()
), (
    "Se han obtenido anchuras relativas negativas."
)


valid_bootstrap_statuses = {
    "calculated",
    "insufficient_sample",
}


assert set(
    neighborhood_statistics[
        "bootstrap_status"
    ].dropna().unique()
).issubset(valid_bootstrap_statuses), (
    "Se ha obtenido un estado bootstrap desconocido."
)


# ============================================================
# Resumen
# ============================================================

num_bootstrap_calculated = (
    neighborhood_statistics[
        "bootstrap_status"
    ]
    .eq("calculated")
    .sum()
)

num_bootstrap_insufficient = (
    neighborhood_statistics[
        "bootstrap_status"
    ]
    .eq("insufficient_sample")
    .sum()
)


print(
    "Intervalos bootstrap calculados:",
    num_bootstrap_calculated,
)

print(
    "Barrios con menos de cinco anuncios:",
    num_bootstrap_insufficient,
)


print(
    "\nBarrios con intervalos relativamente más amplios:"
)


display(
    neighborhood_statistics.loc[
        calculated_bootstrap_mask,
        [
            "zone_key",
            "num_listings",
            "price_m2_median",
            "price_m2_median_ci_low",
            "price_m2_median_ci_high",
            "price_m2_median_ci_width",
            "price_m2_median_ci_relative_width_percentage",
        ],
    ]
    .sort_values(
        [
            "price_m2_median_ci_relative_width_percentage",
            "num_listings",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(15)
)


print(
    "\nBarrios sin intervalo por tener menos de cinco anuncios:"
)


display(
    neighborhood_statistics.loc[
        neighborhood_statistics[
            "bootstrap_status"
        ].eq("insufficient_sample"),
        [
            "zone_key",
            "num_listings",
            "price_m2_median",
            "bootstrap_status",
        ],
    ]
    .sort_values(
        "num_listings"
    )
)

Intervalos bootstrap calculados: 126
Barrios con menos de cinco anuncios: 3

Barrios con intervalos relativamente más amplios:


,zone_key,num_listings,price_m2_median,price_m2_median_ci_low,price_m2_median_ci_high,price_m2_median_ci_width,price_m2_median_ci_relative_width_percentage
97,MAD-158,8,7637.49,4438.64,9044.44,4605.80,60.31
103,MAD-165,12,4203.28,3486.59,6000.49,2513.90,59.81
127,MAD-214,17,4241.25,3639.13,5591.40,1952.27,46.03
87,MAD-144,18,3886.39,3030.76,4485.98,1455.22,37.44
57,MAD-101,24,3443.13,2777.78,4064.81,1287.04,37.38
95,MAD-156,33,6508.47,5304.88,7608.70,2303.82,35.40
71,MAD-121,19,2726.19,2339.13,3235.29,896.16,32.87
128,MAD-215,17,4310.95,3746.59,5102.04,1355.45,31.44
43,MAD-082,21,3929.54,3405.22,4638.10,1232.87,31.37
115,MAD-194,75,3670.21,2941.18,4054.05,1112.88,30.32



Barrios sin intervalo por tener menos de cinco anuncios:


,zone_key,num_listings,price_m2_median,bootstrap_status
85,MAD-142,2,3747.77,insufficient_sample
125,MAD-212,3,6560.98,insufficient_sample
84,MAD-141,4,3709.27,insufficient_sample


## 4. Valorar la fiabilidad del precio de cada barrio

Hasta ahora se han calculado el número de anuncios, la dispersión de los precios y el intervalo de confianza de la mediana. A continuación, se combinará esta información para obtener una medida orientativa de la fiabilidad del precio de cada barrio.

Esta medida tendrá en cuenta tres aspectos:

- cuántos anuncios hay disponibles;
- la amplitud del intervalo de confianza;
- las diferencias de precio entre las viviendas del propio barrio.

Un barrio tendrá una estimación más estable cuando disponga de bastantes anuncios, el intervalo de confianza sea estrecho y los precios no sean excesivamente diferentes entre sí.

La puntuación se utilizará para comparar los barrios de este conjunto de datos. No representa una garantía absoluta sobre el precio real de mercado, sino una forma de indicar qué estimaciones deben interpretarse con mayor precaución.

Los barrios con menos de cinco anuncios se marcarán como insuficientes, ya que en ellos no se ha calculado el intervalo de confianza. Además, aunque se pueda calcular el intervalo, una muestra inferior a quince anuncios seguirá considerándose poco representativa.

In [79]:
# ============================================================
# Puntuación comparativa de fiabilidad del precio
# ============================================================

# La puntuación se calcula únicamente para los barrios en los
# que se ha podido obtener un intervalo bootstrap.

reliability_calculated_mask = (
    neighborhood_statistics[
        "bootstrap_status"
    ].eq("calculated")
)


# Creamos inicialmente las columnas vacías.

reliability_score_columns = [
    "sample_size_score",
    "ci_precision_score",
    "price_homogeneity_score",
    "price_reliability_score",
]

for column in reliability_score_columns:
    neighborhood_statistics[column] = np.nan


# ============================================================
# 1. Puntuación por tamaño de la muestra
# ============================================================

# Los barrios con más anuncios reciben una puntuación superior.
# La puntuación se calcula de forma relativa al resto de barrios.

neighborhood_statistics.loc[
    reliability_calculated_mask,
    "sample_size_score",
] = (
    neighborhood_statistics.loc[
        reliability_calculated_mask,
        "num_listings",
    ]
    .rank(
        method="average",
        pct=True,
    )
    * 100
)


# ============================================================
# 2. Puntuación por precisión del intervalo
# ============================================================

# Una menor anchura relativa del intervalo indica una mediana
# más estable. Por eso se invierte el rango porcentual.

ci_relative_rank = (
    neighborhood_statistics.loc[
        reliability_calculated_mask,
        "price_m2_median_ci_relative_width_percentage",
    ]
    .rank(
        method="average",
        pct=True,
    )
)


neighborhood_statistics.loc[
    reliability_calculated_mask,
    "ci_precision_score",
] = (
    1 - ci_relative_rank
) * 100


# ============================================================
# 3. Puntuación por homogeneidad de precios
# ============================================================

# Un IQR relativo menor indica que los precios del barrio están
# más concentrados alrededor del valor central.

price_dispersion_rank = (
    neighborhood_statistics.loc[
        reliability_calculated_mask,
        "price_m2_relative_iqr_percentage",
    ]
    .rank(
        method="average",
        pct=True,
    )
)


neighborhood_statistics.loc[
    reliability_calculated_mask,
    "price_homogeneity_score",
] = (
    1 - price_dispersion_rank
) * 100


# ============================================================
# Puntuación final
# ============================================================

# El intervalo de confianza recibe el mayor peso porque mide
# directamente la estabilidad de la mediana.
#
# El tamaño de la muestra también tiene un peso importante.
# La dispersión interna se utiliza como información adicional.

neighborhood_statistics.loc[
    reliability_calculated_mask,
    "price_reliability_score",
] = (
    0.40
    * neighborhood_statistics.loc[
        reliability_calculated_mask,
        "ci_precision_score",
    ]
    + 0.35
    * neighborhood_statistics.loc[
        reliability_calculated_mask,
        "sample_size_score",
    ]
    + 0.25
    * neighborhood_statistics.loc[
        reliability_calculated_mask,
        "price_homogeneity_score",
    ]
)


neighborhood_statistics[
    reliability_score_columns
] = (
    neighborhood_statistics[
        reliability_score_columns
    ].round(2)
)


# ============================================================
# Nivel orientativo de fiabilidad
# ============================================================

# Los barrios con menos de cinco anuncios no tienen intervalo.
# Los que tienen entre 5 y 14 anuncios se mantienen en nivel bajo,
# aunque su dispersión resulte favorable.

neighborhood_statistics[
    "price_reliability_level"
] = np.select(
    [
        neighborhood_statistics[
            "bootstrap_status"
        ].eq("insufficient_sample"),

        (
            neighborhood_statistics[
                "bootstrap_status"
            ].eq("calculated")
            & neighborhood_statistics[
                "num_listings"
            ].lt(15)
        ),

        (
            neighborhood_statistics[
                "num_listings"
            ].ge(15)
            & neighborhood_statistics[
                "price_reliability_score"
            ].ge(75)
        ),

        (
            neighborhood_statistics[
                "num_listings"
            ].ge(15)
            & neighborhood_statistics[
                "price_reliability_score"
            ].ge(50)
        ),
    ],
    [
        "insufficient_sample",
        "low",
        "high",
        "medium",
    ],
    default="low",
)


# Añadimos una explicación sencilla del nivel asignado.

def explain_price_reliability(row):
    if row["price_reliability_level"] == "insufficient_sample":
        return (
            "No se calcula una fiabilidad completa porque "
            "hay menos de cinco anuncios."
        )

    if (
        row["num_listings"] < 15
        and row["bootstrap_status"] == "calculated"
    ):
        return (
            "El intervalo se ha calculado, pero el número "
            "de anuncios sigue siendo reducido."
        )

    if row["price_reliability_level"] == "high":
        return (
            "La estimación presenta una estabilidad alta "
            "en comparación con el resto de barrios."
        )

    if row["price_reliability_level"] == "medium":
        return (
            "La estimación presenta una estabilidad intermedia "
            "en comparación con el resto de barrios."
        )

    return (
        "La estimación debe interpretarse con precaución por "
        "el tamaño de la muestra o la variabilidad de los precios."
    )


neighborhood_statistics[
    "price_reliability_reason"
] = neighborhood_statistics.apply(
    explain_price_reliability,
    axis=1,
)


# ============================================================
# Validaciones
# ============================================================

assert (
    neighborhood_statistics.loc[
        reliability_calculated_mask,
        "price_reliability_score",
    ]
    .between(0, 100)
    .all()
), (
    "Se han obtenido puntuaciones de fiabilidad fuera "
    "del intervalo entre 0 y 100."
)


valid_reliability_levels = {
    "high",
    "medium",
    "low",
    "insufficient_sample",
}


assert set(
    neighborhood_statistics[
        "price_reliability_level"
    ].unique()
).issubset(valid_reliability_levels), (
    "Se ha generado un nivel de fiabilidad desconocido."
)


assert (
    neighborhood_statistics.loc[
        neighborhood_statistics[
            "num_listings"
        ].lt(5),
        "price_reliability_level",
    ]
    .eq("insufficient_sample")
    .all()
), (
    "Los barrios con menos de cinco anuncios no se han "
    "clasificado correctamente."
)


# ============================================================
# Resumen
# ============================================================

print(
    "Distribución de los niveles de fiabilidad:"
)

display(
    neighborhood_statistics[
        "price_reliability_level"
    ]
    .value_counts()
    .rename_axis("price_reliability_level")
    .reset_index(name="num_neighborhoods")
)


print(
    "\nBarrios con mayor puntuación de fiabilidad:"
)

display(
    neighborhood_statistics[
        [
            "zone_key",
            "num_listings",
            "price_m2_median",
            "price_m2_median_ci_relative_width_percentage",
            "price_m2_relative_iqr_percentage",
            "price_reliability_score",
            "price_reliability_level",
        ]
    ]
    .sort_values(
        "price_reliability_score",
        ascending=False,
    )
    .head(15)
)


print(
    "\nBarrios con menor puntuación calculada:"
)

display(
    neighborhood_statistics.loc[
        neighborhood_statistics[
            "price_reliability_score"
        ].notna(),
        [
            "zone_key",
            "num_listings",
            "price_m2_median",
            "price_m2_median_ci_relative_width_percentage",
            "price_m2_relative_iqr_percentage",
            "price_reliability_score",
            "price_reliability_level",
        ],
    ]
    .sort_values(
        "price_reliability_score",
        ascending=True,
    )
    .head(15)
)

Distribución de los niveles de fiabilidad:


,price_reliability_level,num_neighborhoods
0,low,63
1,medium,44
2,high,19
3,insufficient_sample,3



Barrios con mayor puntuación de fiabilidad:


,zone_key,num_listings,price_m2_median,price_m2_median_ci_relative_width_percentage,price_m2_relative_iqr_percentage,price_reliability_score,price_reliability_level
19,MAD-042,601,10420.73,3.03,26.84,91.75,high
22,MAD-045,303,10499.23,5.59,26.36,89.48,high
31,MAD-062,154,6214.17,6.78,21.34,89.01,high
0,MAD-011,297,6614.32,6.58,25.56,88.12,high
15,MAD-034,208,9401.79,5.83,26.43,87.34,high
5,MAD-016,310,8037.53,4.90,30.43,85.83,high
28,MAD-055,127,7660.00,6.73,24.61,85.28,high
27,MAD-054,137,7344.86,8.38,22.29,84.84,high
4,MAD-015,439,7571.43,4.72,32.86,83.81,high
39,MAD-074,226,10604.45,7.67,28.53,83.17,high



Barrios con menor puntuación calculada:


,zone_key,num_listings,price_m2_median,price_m2_median_ci_relative_width_percentage,price_m2_relative_iqr_percentage,price_reliability_score,price_reliability_level
97,MAD-158,8,7637.49,60.31,55.59,1.35,low
103,MAD-165,12,4203.28,59.81,55.23,2.56,low
127,MAD-214,17,4241.25,46.03,46.03,4.92,low
87,MAD-144,18,3886.39,37.44,41.51,7.60,low
57,MAD-101,24,3443.13,37.38,44.04,10.46,low
71,MAD-121,19,2726.19,32.87,36.99,10.58,low
95,MAD-156,33,6508.47,35.40,49.23,12.22,low
123,MAD-208,29,4951.92,29.26,44.74,14.01,low
72,MAD-122,38,3411.29,29.69,48.36,15.81,low
128,MAD-215,17,4310.95,31.44,31.44,17.02,low


## 5. Incorporar los 131 barrios oficiales

Las estadísticas anteriores solo contienen los barrios que disponen de al menos un anuncio válido. Sin embargo, la tabla final debe mantener los 131 barrios oficiales de Madrid.

Por este motivo, los resultados se unirán con la tabla territorial completa. Los barrios sin anuncios seguirán apareciendo, pero sus indicadores inmobiliarios quedarán vacíos y se marcarán claramente como zonas sin datos.

De esta forma, la ausencia de información no se confunde con un precio igual a cero y será posible añadir posteriormente otras fuentes, como los tiempos de desplazamiento o los precios oficiales.

In [80]:
# ============================================================
# Unión de la información territorial, la cobertura y las
# estadísticas inmobiliarias
# ============================================================

coverage_columns_to_merge = [
    "zone_key",
    "total_listings",
    "eligible_listings",
    "excluded_listings",
    "eligible_percentage",
    "excluded_percentage",
    "market_share_percentage",
    "coverage_status",
]


neighborhood_real_estate = (
    zones[
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
        ]
    ]
    .merge(
        coverage_by_zone[
            coverage_columns_to_merge
        ],
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        neighborhood_statistics,
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# Tratamiento de los barrios sin anuncios
# ============================================================

# Los recuentos pueden rellenarse con cero.
# Las medias, medianas y proporciones permanecen como NaN porque
# no existe información con la que calcularlas.

count_columns = [
    "total_listings",
    "eligible_listings",
    "excluded_listings",
    "num_listings",
    "rooms_known_count",
    "rooms_unknown_count",
    "bathrooms_known_count",
    "bathrooms_unknown_count",
    "elevator_known_count",
    "elevator_unknown_count",
    "exterior_known_count",
    "exterior_unknown_count",
    "bootstrap_sample_size",
]


for column in count_columns:
    neighborhood_real_estate[column] = (
        neighborhood_real_estate[column]
        .fillna(0)
        .astype(int)
    )


# Las condiciones utilizadas para el bootstrap se mantienen como
# información metodológica en todas las filas.

neighborhood_real_estate[
    "bootstrap_iterations"
] = (
    neighborhood_real_estate[
        "bootstrap_iterations"
    ]
    .fillna(BOOTSTRAP_ITERATIONS)
    .astype(int)
)

neighborhood_real_estate[
    "bootstrap_confidence_level"
] = (
    neighborhood_real_estate[
        "bootstrap_confidence_level"
    ]
    .fillna(BOOTSTRAP_CONFIDENCE_LEVEL)
)


# Indicadores generales de disponibilidad.

neighborhood_real_estate[
    "has_real_estate_data"
] = (
    neighborhood_real_estate[
        "num_listings"
    ].gt(0)
)

neighborhood_real_estate[
    "price_estimate_available"
] = (
    neighborhood_real_estate[
        "price_m2_median"
    ].notna()
)


# Estados específicos de los barrios sin anuncios.

no_real_estate_data_mask = (
    ~neighborhood_real_estate[
        "has_real_estate_data"
    ]
)

neighborhood_real_estate.loc[
    no_real_estate_data_mask,
    "bootstrap_status",
] = "no_data"

neighborhood_real_estate.loc[
    no_real_estate_data_mask,
    "price_reliability_level",
] = "no_data"

neighborhood_real_estate.loc[
    no_real_estate_data_mask,
    "price_reliability_reason",
] = (
    "No hay anuncios elegibles para calcular "
    "indicadores inmobiliarios."
)


# ============================================================
# Porcentajes de viviendas con ascensor y exteriores
# ============================================================

# Se conservan los ratios entre 0 y 1 para el futuro modelo,
# pero se añaden porcentajes para facilitar su interpretación.

neighborhood_real_estate[
    "elevator_percentage"
] = (
    neighborhood_real_estate[
        "elevator_ratio"
    ]
    * 100
).round(2)

neighborhood_real_estate[
    "exterior_percentage"
] = (
    neighborhood_real_estate[
        "exterior_ratio"
    ]
    * 100
).round(2)


# ============================================================
# Información sobre el alcance de los datos
# ============================================================

# Estas columnas recuerdan que los valores proceden de anuncios
# de venta y representan precios de oferta, no precios finales
# de compraventa.

neighborhood_real_estate[
    "market_scope"
] = "residential_sale"

neighborhood_real_estate[
    "price_type"
] = "asking_price"

neighborhood_real_estate[
    "source_snapshot_date_status"
] = "unknown"


# ============================================================
# Orden territorial
# ============================================================

neighborhood_real_estate = (
    neighborhood_real_estate
    .sort_values(
        [
            "district_name",
            "neighborhood_name",
        ]
    )
    .reset_index(drop=True)
)


# IMPORTANTE:
# Después de ordenar y reiniciar el índice, volvemos a crear
# las máscaras para que coincidan con el nuevo orden de las filas.

no_real_estate_data_mask = (
    ~neighborhood_real_estate[
        "has_real_estate_data"
    ]
)

real_estate_data_mask = (
    neighborhood_real_estate[
        "has_real_estate_data"
    ]
)


# ============================================================
# Comprobaciones
# ============================================================

assert len(neighborhood_real_estate) == 131, (
    "La tabla final no contiene los 131 barrios oficiales."
)

assert not neighborhood_real_estate[
    "zone_key"
].duplicated().any(), (
    "Existen barrios duplicados en la tabla final."
)

assert (
    neighborhood_real_estate[
        "num_listings"
    ]
    == neighborhood_real_estate[
        "eligible_listings"
    ]
).all(), (
    "El número de anuncios agregado no coincide con "
    "la cobertura elegible."
)


# Los barrios sin anuncios no deben tener un precio calculado.

assert (
    neighborhood_real_estate.loc[
        no_real_estate_data_mask,
        "price_m2_median",
    ]
    .isna()
    .all()
), (
    "Algún barrio sin anuncios tiene un precio asignado."
)


# Todos los barrios con anuncios deben tener una mediana.

assert (
    neighborhood_real_estate.loc[
        real_estate_data_mask,
        "price_m2_median",
    ]
    .notna()
    .all()
), (
    "Algún barrio con anuncios no tiene mediana de precio."
)


# Los estados de los barrios sin datos deben estar correctamente
# identificados.

assert (
    neighborhood_real_estate.loc[
        no_real_estate_data_mask,
        "bootstrap_status",
    ]
    .eq("no_data")
    .all()
), (
    "Algún barrio sin anuncios no tiene bootstrap_status = no_data."
)

assert (
    neighborhood_real_estate.loc[
        no_real_estate_data_mask,
        "price_reliability_level",
    ]
    .eq("no_data")
    .all()
), (
    "Algún barrio sin anuncios no tiene "
    "price_reliability_level = no_data."
)


# ============================================================
# Resumen
# ============================================================

print(
    "Filas de la tabla inmobiliaria:",
    len(neighborhood_real_estate),
)

print(
    "Barrios con estadísticas:",
    real_estate_data_mask.sum(),
)

print(
    "Barrios sin estadísticas:",
    no_real_estate_data_mask.sum(),
)

print(
    "Total de anuncios agregados:",
    neighborhood_real_estate[
        "num_listings"
    ].sum(),
)


print(
    "\nBarrios sin información inmobiliaria:"
)

display(
    neighborhood_real_estate.loc[
        no_real_estate_data_mask,
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
            "total_listings",
            "eligible_listings",
            "coverage_status",
            "bootstrap_status",
            "price_reliability_level",
        ],
    ]
)

Filas de la tabla inmobiliaria: 131
Barrios con estadísticas: 129
Barrios sin estadísticas: 2
Total de anuncios agregados: 11109

Barrios sin información inmobiliaria:


,zone_key,neighborhood_name,district_name,total_listings,eligible_listings,coverage_status,bootstrap_status,price_reliability_level
1,MAD-027,Atocha,Arganzuela,0,0,no_listings,no_data,no_data
121,MAD-192,Valdebernardo,Vicálvaro,0,0,no_listings,no_data,no_data


### 5.1. Preparar la tabla final

Antes de guardar el resultado, se organizarán las columnas para que la tabla sea más fácil de utilizar.

En primer lugar aparecerá la identificación del barrio y la información sobre el número de anuncios. Después se incluirán los precios, los intervalos de confianza, la superficie y las características de las viviendas.

Las últimas columnas recogerán la puntuación de fiabilidad y la información necesaria para recordar el alcance y las limitaciones de la fuente utilizada.

In [81]:
# ============================================================
# Orden de las columnas de la tabla final
# ============================================================

final_column_order = [
    # Identificación territorial
    "zone_key",
    "neighborhood_name",
    "district_name",

    # Disponibilidad y cobertura general
    "has_real_estate_data",
    "price_estimate_available",
    "coverage_status",
    "total_listings",
    "eligible_listings",
    "excluded_listings",
    "eligible_percentage",
    "excluded_percentage",
    "market_share_percentage",
    "num_listings",

    # Precio total
    "price_eur_mean",
    "price_eur_median",
    "price_eur_q1",
    "price_eur_q3",
    "price_eur_iqr",
    "price_eur_min",
    "price_eur_max",

    # Precio por metro cuadrado
    "price_m2_mean",
    "price_m2_median",
    "price_m2_q1",
    "price_m2_q3",
    "price_m2_iqr",
    "price_m2_std",
    "price_m2_relative_iqr_percentage",

    # Incertidumbre del precio
    "price_m2_median_ci_low",
    "price_m2_median_ci_high",
    "price_m2_median_ci_width",
    "price_m2_median_ci_relative_width_percentage",
    "price_m2_median_bootstrap_se",
    "bootstrap_sample_size",
    "bootstrap_iterations",
    "bootstrap_confidence_level",
    "bootstrap_status",

    # Superficie
    "area_m2_mean",
    "area_m2_median",
    "area_m2_q1",
    "area_m2_q3",
    "area_m2_iqr",
    "area_m2_min",
    "area_m2_max",

    # Habitaciones
    "rooms_mean",
    "rooms_median",
    "rooms_known_count",
    "rooms_unknown_count",
    "rooms_coverage_percentage",

    # Baños
    "bathrooms_mean",
    "bathrooms_median",
    "bathrooms_known_count",
    "bathrooms_unknown_count",
    "bathrooms_coverage_percentage",

    # Ascensor
    "elevator_ratio",
    "elevator_percentage",
    "elevator_known_count",
    "elevator_unknown_count",
    "elevator_coverage_percentage",

    # Vivienda exterior
    "exterior_ratio",
    "exterior_percentage",
    "exterior_known_count",
    "exterior_unknown_count",
    "exterior_coverage_percentage",

    # Fiabilidad
    "sample_size_score",
    "ci_precision_score",
    "price_homogeneity_score",
    "price_reliability_score",
    "price_reliability_level",
    "price_reliability_reason",

    # Alcance de la fuente
    "market_scope",
    "price_type",
    "source_snapshot_date_status",
]


missing_final_columns = [
    column
    for column in final_column_order
    if column not in neighborhood_real_estate.columns
]

if missing_final_columns:
    raise ValueError(
        "Faltan columnas en la tabla final: "
        f"{missing_final_columns}"
    )


neighborhood_real_estate = (
    neighborhood_real_estate[
        final_column_order
    ]
)


print(
    "Número de columnas de la tabla final:",
    neighborhood_real_estate.shape[1],
)

display(
    neighborhood_real_estate.head()
)

Número de columnas de la tabla final: 72


,zone_key,neighborhood_name,district_name,has_real_estate_data,price_estimate_available,coverage_status,total_listings,eligible_listings,excluded_listings,eligible_percentage,...,exterior_coverage_percentage,sample_size_score,ci_precision_score,price_homogeneity_score,price_reliability_score,price_reliability_level,price_reliability_reason,market_scope,price_type,source_snapshot_date_status
0,MAD-022,Acacias,Arganzuela,True,True,available,67,66,1,98.51,...,96.97,50.79,45.24,47.22,47.68,low,La estimación debe interpretarse con precaució...,residential_sale,asking_price,unknown
1,MAD-027,Atocha,Arganzuela,False,False,no_listings,0,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,no_data,No hay anuncios elegibles para calcular indica...,residential_sale,asking_price,unknown
2,MAD-023,Chopera,Arganzuela,True,True,available,35,34,1,97.14,...,100.00,28.57,71.43,97.62,62.98,medium,La estimación presenta una estabilidad interme...,residential_sale,asking_price,unknown
3,MAD-025,Delicias,Arganzuela,True,True,available,68,60,8,88.24,...,95.00,48.41,84.92,92.06,73.93,medium,La estimación presenta una estabilidad interme...,residential_sale,asking_price,unknown
4,MAD-021,Imperial,Arganzuela,True,True,available,73,71,2,97.26,...,98.59,56.35,92.06,47.22,68.35,medium,La estimación presenta una estabilidad interme...,residential_sale,asking_price,unknown


## 6. Comprobación y guardado de los resultados

Antes de guardar los archivos, se realizará una última revisión para comprobar que:

- aparecen los 131 barrios oficiales;
- no hay barrios repetidos;
- el número total de anuncios coincide con el dataset de entrada;
- los barrios con anuncios tienen un precio calculado;
- los barrios sin anuncios permanecen claramente identificados;
- los porcentajes y proporciones se encuentran dentro de valores válidos.

Finalmente, se guardarán tres archivos: la auditoría de cobertura, el resumen inmobiliario completo y una tabla específica con la incertidumbre de las estimaciones de precio.

In [82]:
# ============================================================
# Validaciones finales
# ============================================================

final_quality_checks = {
    "contains_131_official_neighborhoods": (
        len(neighborhood_real_estate) == 131
    ),

    "zone_key_is_unique": (
        not neighborhood_real_estate[
            "zone_key"
        ].duplicated().any()
    ),

    "all_zone_keys_are_present": (
        neighborhood_real_estate[
            "zone_key"
        ].notna().all()
    ),

    "total_listings_match_clean_dataset": (
        neighborhood_real_estate[
            "total_listings"
        ].sum()
        == len(listings_clean)
    ),

    "eligible_listings_match_market_dataset": (
        neighborhood_real_estate[
            "eligible_listings"
        ].sum()
        == len(listings_market)
    ),

    "aggregated_listings_match_market_dataset": (
        neighborhood_real_estate[
            "num_listings"
        ].sum()
        == len(listings_market)
    ),

    "aggregated_and_eligible_counts_match": (
        neighborhood_real_estate[
            "num_listings"
        ].equals(
            neighborhood_real_estate[
                "eligible_listings"
            ]
        )
    ),

    "neighborhoods_with_data_have_price": (
        neighborhood_real_estate.loc[
            neighborhood_real_estate[
                "has_real_estate_data"
            ],
            "price_m2_median",
        ]
        .notna()
        .all()
    ),

    "neighborhoods_without_data_have_no_price": (
        neighborhood_real_estate.loc[
            ~neighborhood_real_estate[
                "has_real_estate_data"
            ],
            "price_m2_median",
        ]
        .isna()
        .all()
    ),

    "elevator_ratios_are_valid": (
        neighborhood_real_estate[
            "elevator_ratio"
        ]
        .dropna()
        .between(0, 1)
        .all()
    ),

    "exterior_ratios_are_valid": (
        neighborhood_real_estate[
            "exterior_ratio"
        ]
        .dropna()
        .between(0, 1)
        .all()
    ),

    "reliability_scores_are_valid": (
        neighborhood_real_estate[
            "price_reliability_score"
        ]
        .dropna()
        .between(0, 100)
        .all()
    ),
}


quality_check_results = pd.DataFrame(
    {
        "check": list(
            final_quality_checks.keys()
        ),
        "passed": list(
            final_quality_checks.values()
        ),
    }
)


display(quality_check_results)


failed_quality_checks = (
    quality_check_results.loc[
        ~quality_check_results["passed"],
        "check",
    ]
    .tolist()
)


if failed_quality_checks:
    raise AssertionError(
        "Han fallado las siguientes comprobaciones: "
        f"{failed_quality_checks}"
    )


# ============================================================
# Tabla específica de incertidumbre del precio
# ============================================================

price_uncertainty_columns = [
    "zone_key",
    "neighborhood_name",
    "district_name",
    "num_listings",
    "price_m2_median",
    "price_m2_q1",
    "price_m2_q3",
    "price_m2_iqr",
    "price_m2_relative_iqr_percentage",
    "price_m2_median_ci_low",
    "price_m2_median_ci_high",
    "price_m2_median_ci_width",
    "price_m2_median_ci_relative_width_percentage",
    "price_m2_median_bootstrap_se",
    "bootstrap_status",
    "price_reliability_score",
    "price_reliability_level",
    "price_reliability_reason",
]


neighborhood_price_uncertainty = (
    neighborhood_real_estate[
        price_uncertainty_columns
    ].copy()
)


# ============================================================
# Rutas de salida
# ============================================================

coverage_path = (
    PROCESSED_DIR
    / "neighborhood_coverage_audit.csv"
)

summary_path = (
    PROCESSED_DIR
    / "neighborhood_real_estate_summary.csv"
)

uncertainty_path = (
    PROCESSED_DIR
    / "neighborhood_price_uncertainty.csv"
)


# ============================================================
# Guardado
# ============================================================

coverage_by_zone.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

neighborhood_real_estate.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

neighborhood_price_uncertainty.to_csv(
    uncertainty_path,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# Resumen final
# ============================================================

print(
    "Todas las comprobaciones finales "
    "se han superado correctamente."
)

print(
    "\nAuditoría de cobertura guardada en:"
)
print(coverage_path)

print(
    "\nResumen inmobiliario guardado en:"
)
print(summary_path)

print(
    "\nAnálisis de incertidumbre guardado en:"
)
print(uncertainty_path)

print(
    "\nResumen del resultado:"
)

print(
    f"- Barrios oficiales: "
    f"{len(neighborhood_real_estate)}"
)

print(
    f"- Barrios con datos inmobiliarios: "
    f"{neighborhood_real_estate['has_real_estate_data'].sum()}"
)

print(
    f"- Barrios sin datos inmobiliarios: "
    f"{(~neighborhood_real_estate['has_real_estate_data']).sum()}"
)

print(
    f"- Anuncios agregados: "
    f"{neighborhood_real_estate['num_listings'].sum():,}"
)

print(
    f"- Columnas del resumen: "
    f"{neighborhood_real_estate.shape[1]}"
)

,check,passed
0,contains_131_official_neighborhoods,True
1,zone_key_is_unique,True
2,all_zone_keys_are_present,True
3,total_listings_match_clean_dataset,True
4,eligible_listings_match_market_dataset,True
5,aggregated_listings_match_market_dataset,True
6,aggregated_and_eligible_counts_match,True
7,neighborhoods_with_data_have_price,True
8,neighborhoods_without_data_have_no_price,True
9,elevator_ratios_are_valid,True


Todas las comprobaciones finales se han superado correctamente.

Auditoría de cobertura guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\neighborhood_coverage_audit.csv

Resumen inmobiliario guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\neighborhood_real_estate_summary.csv

Análisis de incertidumbre guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\neighborhood_price_uncertainty.csv

Resumen del resultado:
- Barrios oficiales: 131
- Barrios con datos inmobiliarios: 129
- Barrios sin datos inmobiliarios: 2
- Anuncios agregados: 11,109
- Columnas del resumen: 72


## 7. Resultado del notebook

El resultado principal de este notebook es `neighborhood_real_estate_summary.csv`, que contiene una fila para cada uno de los 131 barrios oficiales de Madrid.

La tabla reúne la información inmobiliaria disponible, la cobertura de cada característica y diferentes medidas que permiten valorar la estabilidad de los precios calculados.

Los barrios sin anuncios se conservan, pero aparecen sin precio y con un estado que indica que no existe información disponible. De esta forma, podrán incorporarse posteriormente datos procedentes de otras fuentes sin perder ninguna zona.

Esta tabla se utilizará en los siguientes pasos para combinar el coste de la vivienda con los tiempos de desplazamiento y construir el sistema de recomendación.